# The perceptron trick in code

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

X, y = make_classification(n_samples=100, n_features=2, n_informative=1, n_redundant=0, n_classes=2,
                           n_clusters_per_class=1, random_state=41, hypercube=False, class_sep=10)
print(X.shape, np.bincount(y))

(100, 2) [50 50]


## The algorithm

In [2]:
def step(z):
    return 1 if z > 0 else 0

def perceptron(X, y, lr=0.1, loops=1000, seed=0):
    rng = np.random.default_rng(seed)       # fixed seed: the same random picks every run
    X = np.insert(X, 0, 1, axis=1)          # column of 1s for the intercept
    weights = np.ones(X.shape[1])
    for _ in range(loops):
        j = rng.integers(0, X.shape[0])
        y_hat = step(X[j] @ weights)
        weights = weights + lr * (y[j] - y_hat) * X[j]
    return weights[0], weights[1:]

intercept_, coef_ = perceptron(X, y)
print("intercept", intercept_, "coefficients", coef_)

intercept 1.0 coefficients [1.3423777  0.19109935]


## From w0 + w1 x1 + w2 x2 = 0 to x2 = m x1 + b

In [3]:
m = -coef_[0] / coef_[1]
b = -intercept_ / coef_[1]
print(f"x2 = {m:.2f} x1 + {b:.2f}")
Xb = np.insert(X, 0, 1, axis=1)
w = np.r_[intercept_, coef_]
print("misclassified:", ((Xb @ w > 0).astype(int) != y).sum())

x2 = -7.02 x1 + -5.23
misclassified: 0


## Five random orders vs logistic regression: distance from the line to the nearest point of each class

In [4]:
def gaps(w):
    d = (Xb @ w) / np.linalg.norm(w[1:])
    return round(d[y == 1].min(), 3), round(-d[y == 0].max(), 3)

for seed in range(5):
    i, c = perceptron(X, y, seed=seed)
    print("perceptron seed", seed, gaps(np.r_[i, c]))
lr = LogisticRegression(C=100, max_iter=10000).fit(X, y)
print("logistic regression ", gaps(np.r_[lr.intercept_, lr.coef_[0]]))

perceptron seed 0 (np.float64(0.127), np.float64(0.065))
perceptron seed 1 (np.float64(0.234), np.float64(0.052))
perceptron seed 2 (np.float64(0.021), np.float64(0.164))
perceptron seed 3 (np.float64(0.02), np.float64(0.104))
perceptron seed 4 (np.float64(0.015), np.float64(0.171))
logistic regression  (np.float64(0.117), np.float64(0.143))


## Test on real data: does a line that hugs one class do worse on new points?
Iris, setosa against versicolor, using sepal length and sepal width (standardised). These two species are linearly separable. 100 times: train on 10 random flowers per class, test on the other 80. Five perceptron runs (seeds 0 to 4) against logistic regression (C=10000).

In [5]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
iris = load_iris()
keep = iris.target < 2                                   # setosa (0) and versicolor (1)
Xi = StandardScaler().fit_transform(iris.data[keep][:, [0, 1]])
yi = iris.target[keep]
perc_err, lr_err = [], []
for ds in range(100):
    r = np.random.default_rng(ds)
    tr = np.concatenate([r.choice(np.where(yi == c)[0], 10, replace=False) for c in (0, 1)])
    te = np.setdiff1d(np.arange(len(yi)), tr)
    runs = [perceptron(Xi[tr], yi[tr], seed=s) for s in range(5)]
    At = np.insert(Xi[te], 0, 1, axis=1)
    perc_err.append(np.mean([np.mean(((At @ np.r_[i, c]) > 0).astype(int) != yi[te]) for i, c in runs]))
    lr_err.append(np.mean(LogisticRegression(C=1e4, max_iter=10000).fit(Xi[tr], yi[tr]).predict(Xi[te]) != yi[te]))
print(f"average test error: perceptron {100 * np.mean(perc_err):.1f}%, logistic regression {100 * np.mean(lr_err):.1f}%")

average test error: perceptron 5.5%, logistic regression 1.2%


In [6]:
lr1 = LogisticRegression().fit(X, y)                       # default C=1
d = (Xb @ np.r_[lr1.intercept_, lr1.coef_[0]]) / np.linalg.norm(lr1.coef_[0])
wrong = (d > 0).astype(int) != y
print("C=1: misclassified", wrong.sum(), "distance to line", np.abs(d[wrong]).round(3))

C=1: misclassified 1 distance to line [0.009]
